<a href="https://colab.research.google.com/github/quarcs-lab/geo4bolivia/blob/main/notebooks/three_packages.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# One question, three packages: did Bolivia's municipalities converge?

Part of the [GeoDevelopment Observatory of Bolivia](https://quarcs-lab.github.io/geo4bolivia/).

The observatory has three Python packages that work as one workflow:

| Step | Package | Role |
| --- | --- | --- |
| 1 | [**geo4bolivia**](https://quarcs-lab.github.io/geo4bolivia/tools/geo4bolivia-package/) | Find, load, understand and prepare the data |
| 2 | [**expdpy**](https://quarcs-lab.github.io/geo4bolivia/tools/expdpy-package/) | Explore and model it as a panel |
| 3 | [**geometrics**](https://quarcs-lab.github.io/geo4bolivia/tools/geometrics-package/) | Analyse its geography: clusters, convergence, inequality |

This notebook takes one question through all three: **did the Sustainable Development Index (IDS) of Bolivia's 343 municipalities and indigenous autonomies converge between ~2012 and ~2024?** The data are the [Atlas de los ODS en Bolivia 2026](https://atlasods.sdsnbolivia.org/) (Andersen & Argandoña, SDSN Bolivia).

## 0. Setup

In Google Colab the next cell installs what Colab lacks (a few seconds). Locally, `uv sync` has installed everything already.

In [1]:
import importlib.util

missing = [p for p in ["geo4bolivia", "expdpy", "geometrics"] if importlib.util.find_spec(p) is None]
if missing:
    !pip install -q {" ".join(missing)}

In [2]:
import warnings

import numpy as np
import plotly.express as px

import geo4bolivia as geo
import expdpy as ex
import geometrics as gm

warnings.filterwarnings("ignore", category=UserWarning)   # keep the notebook readable
print("geo4bolivia", geo.__version__, "| expdpy", ex.__version__, "| geometrics", gm.__version__)

geo4bolivia 0.5.0 | expdpy 0.5.2 | geometrics 0.1.3


## 1. geo4bolivia: find, load and understand the data

### Find the variable

We do not need to know the dataset in advance: `geo.search()` looks through every variable of every dataset.

In [3]:
geo.search("sustainable development index")[["dataset", "variable", "label", "years", "units"]]

,dataset,variable,label,years,units
0,sdg,index_sdg,"Sustainable Development Index, IDS (0-100)","2012, 2024",343


`geo.info()` tells what the `sdg` dataset is, where it comes from, its license and how to cite it:

In [4]:
geo.info("sdg")

Sustainable Development Index, IDS (SDG Atlas of Bolivia 2026)  [sdg]

  The Sustainable Development Index (IDS) of the Atlas de los ODS en Bolivia 2026 (SDSN
  Bolivia), which summarises 81 indicators of 15 Sustainable Development Goals, and its national
  ranking, for Bolivia's 343 municipalities and indigenous autonomies (GAIOC), near the 2012
  census and near the 2024 census (stored as 2012 and 2024), with the census population of both
  years and the 2024 urbanization rate as context.

Coverage
  - 343 municipality units, 2012, 2024, 4 variables

Actual years behind the stored years (geo.reference_years)
  - 2012: ~2012; 2024: ~2024  (2 variables)
  - 2012: 2012; 2024: 2024  (1 variables)
  - 2024: 2024  (1 variables)

License of the published files
  - Creative Commons Attribution 4.0 International License

Sources
  - Andersen, L. E., & Argandoña, F. (2026). Atlas de los ODS en Bolivia 2026: Brechas y
  tendencias del desarrollo sostenible en las Entidades Territoriales Autónomas, 2012–2024. La
  Paz: Universidad Privada Boliviana, SDSN Bolivia.
    License: No open license stated (© Universidad Privada Boliviana); used and shared with the
    permission of SDSN Bolivia
  - SDSN Bolivia (2025). Límites Municipales Bolivia 2025 [Shapefile], with the population of
  the 2012 and 2024 Population and Housing Censuses (INE). https://sdsnbolivia.org/datos-
  espaciales/
    License: No open license stated (© Universidad Privada Boliviana); used and shared with the
    permission of SDSN Bolivia

Cite as (geo.cite)
  GeoDevelopment Observatory of Bolivia (2026). Sustainable Development Index (IDS), from the
  Atlas de los ODS en Bolivia 2026 (343 municipalities, ~2012 and ~2024).
  https://github.com/quarcs-lab/geo4bolivia/blob/main/docs/datasets/sdg.md

Notes
  - Both periods are stored at nominal years: 2012 for the Atlas's initial values (~2012) and
  2024 for its final values (~2024). Each indicator's actual reference years differ (e.g. 2008
  and 2023 for the health survey, three-year averages for volatile indicators) and are given in
  its description and in `geo:reference_years`.
  - Load one variable in both periods, e.g. `geo.load("sdg", variables="index_sdg",
  names=True)`, and its change per unit with `geo.change(df, "index_sdg")`.
  - The units are the 343 municipalities and indigenous autonomies (GAIOC) of 2025. Four of them
  were created after the 2012 census (TIOC-Raqaypampa, San Pedro de Macha, TIOC-Jatun Ayllu Yura
  and TIOC-Territorio Indígena Multiétnico); the Atlas gives them values in both periods.
  - `population` sums; the IDS and `urbanization` aggregate as census-population-weighted means
  (`geo.aggregate(geo.load("sdg"), to="department", dataset="sdg")`), approximations rather than
  the Atlas's own department figures; the ranking is not additive (`none`).
  - The goal indices and the 81 indicators are in the goal datasets `sdg01` ... `sdg17` (no 12
  or 14).

Notebooks
  - Dynamics of sustainable development in Bolivia, 2012–2024
  - Load, link and build Bolivian indicators with geo4bolivia
  - Exploratory data analysis of multidimensional poverty
  - Spatial inequality of poverty and night-time lights
  - Poverty in Bolivia: an extended EDA and ESDA

Documentation: https://github.com/quarcs-lab/geo4bolivia/blob/main/docs/datasets/sdg.md

### Load and describe it

`geo.load()` returns one row per municipality and period, here with the municipality's name and its department. We also load the census population, used later as a weight and a regressor.

In [5]:
ids = geo.load("sdg", variables=["index_sdg", "population"], groups=["department"])
ids.head()

,geo_id,name,year,department,index_sdg,population
0,BO-01-01-01,Azurduy,2012,Chuquisaca,40.8,10652
1,BO-01-01-01,Azurduy,2024,Chuquisaca,54.5,7903
2,BO-01-01-02,Tarvita,2012,Chuquisaca,44.5,14261
3,BO-01-01-02,Tarvita,2024,Chuquisaca,53.9,16812
4,BO-01-02-01,Villa Serrano,2012,Chuquisaca,53.6,11161


In [6]:
geo.describe(ids)

,variable,label,unit,year,count,mean,std,min,p25,median,p75,max
0,index_sdg,"Sustainable Development Index, IDS (0-100)",index (0-100),2012,343,50.123032,5.819148,37.2,45.95,49.6,53.35,70.5
1,index_sdg,"Sustainable Development Index, IDS (0-100)",index (0-100),2024,343,60.081341,5.263740,47.3,56.40,59.8,63.70,77.3
2,population,Population (census),people,2012,343,29329.026239,107270.841863,509.0,5945.00,10647.0,19544.00,1444425.0
3,population,Population (census),people,2024,343,33133.915452,116499.083621,875.0,6691.50,12296.0,21083.00,1610982.0


The mean rose and the spread fell: a first hint of convergence.

### Change per municipality

`geo.change()` gives each unit's initial and final value, its change and its rank in each period:

In [7]:
chg = geo.change(ids, "index_sdg")
print(f"The IDS rose in {(chg['index_sdg_change'] > 0).sum()} of {len(chg)} units; "
      f"correlation of the change with the initial level: "
      f"{np.corrcoef(chg['index_sdg_2012'], chg['index_sdg_change'])[0, 1]:.2f}")
px.scatter(chg, x="index_sdg_2012", y="index_sdg_change", color="department", hover_name="name",
           labels=geo.labels(chg), title="Did the municipalities that started lower gain more?")

The IDS rose in 342 of 343 units; correlation of the change with the initial level: -0.43


### Departments

The SDG indicators aggregate as census-population-weighted means; the package fetches the population itself. These are approximations, not the Atlas's own department figures.

In [8]:
dep = geo.add_names(geo.aggregate(ids[["geo_id", "year", "index_sdg"]], to="department", dataset="sdg"))
px.bar(dep, x="name", y="index_sdg", color=dep["year"].astype(str), barmode="group",
       labels={**geo.labels(dep), "color": "Period"},
       title="Sustainable Development Index by department (census-population weighted)")

## 2. expdpy: explore and model the panel

`geo.for_expdpy()` hands the panel over in expdpy's format, with a data dictionary built from the metadata; `ex.set_labels()` declares the unit (`geo_id`) and time (`year`) dimensions. We add the log of the census population.

In [9]:
ids["ln_pop"] = np.log(ids["population"])
df, df_def = geo.for_expdpy(ids, labels={"ln_pop": "Log of census population"})
df = ex.set_labels(df, df_def, set_panel=True)
df_def

,var_name,var_def,label,type,role,can_be_na
0,geo_id,geo ID,geo ID,entity,,False
1,name,Name,Unit name,factor,entity_name,True
2,year,Year,Year,time,,False
3,department,Name of the Department the unit lies in,Department name,factor,,True
4,index_sdg,"The Sustainable Development Index (IDS, Índice...","Sustainable Development Index, IDS (0-100)",numeric,,True
5,population,Population counted by the Population and Housi...,Population (census),numeric,,True
6,ln_pop,Log of census population,Log of census population,numeric,,True


### Between and within variation

How much of the IDS's variation is differences between municipalities, and how much is change within them over time?

In [10]:
xt = ex.explore_xtsum_table(df, var=["index_sdg"])
print(xt.interpret())
xt.gt

Splitting each variable's variation into **between** (differences across units) and **within** (variation over time inside a unit):
- **index_sdg**: between SD 5.35, within SD 5.19 — variation is split between cross-unit and over-time.

_Between variation drives cross-sectional comparisons; within variation is what fixed-effects (within) estimators rely on._


GT(_tbl_data=                                     variable Statistic       Mean  Std. dev.  \
0  Sustainable Development Index, IDS (0-100)   Overall  55.102187   7.454391   
1  Sustainable Development Index, IDS (0-100)   Between        NaN   5.353554   
2  Sustainable Development Index, IDS (0-100)    Within        NaN   5.191267   

        Min.       Max.      N  n units  T-bar  
0  37.200000  77.300000  686.0    343.0    2.0  
1  43.300000  73.900000    NaN      NaN    NaN  
2  44.852187  65.352187    NaN      NaN    NaN  , _body=<great_tables._gt_data.Body object at 0x1251c9f10>, _boxhead=Boxhead([ColInfo(var='variable', type=<ColInfoTypeEnum.row_group: 3>, column_label='variable', column_align='left', column_width=None, stub_level=None), ColInfo(var='Statistic', type=<ColInfoTypeEnum.stub: 2>, column_label='Statistic', column_align='left', column_width=None, stub_level=0), ColInfo(var='Mean', type=<ColInfoTypeEnum.default: 1>, column_label='Mean', column_align='right', column_width=None, stub_level=None), ColInfo(var='Std. dev.', type=<ColInfoTypeEnum.default: 1>, column_label='Std. dev.', column_align='right', column_width=None, stub_level=None), ColInfo(var='Min.', type=<ColInfoTypeEnum.default: 1>, column_label='Min.', column_align='right', column_width=None, stub_level=None), ColInfo(var='Max.', type=<ColInfoTypeEnum.default: 1>, column_label='Max.', column_align='right', column_width=None, stub_level=None), ColInfo(var='N', type=<ColInfoTypeEnum.default: 1>, column_label='N', column_align='right', column_width=None, stub_level=None), ColInfo(var='n units', type=<ColInfoTypeEnum.default: 1>, column_label='n units', column_align='right', column_width=None, stub_level=None), ColInfo(var='T-bar', type=<ColInfoTypeEnum.default: 1>, column_label='T-bar', column_align='right', column_width=None, stub_level=None)]), _stub=<great_tables._gt_data.Stub object at 0x1251cf450>, _spanners=Spanners([]), _heading=Heading(title='Within/Between Variation (xtsum)', subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x1251cac90>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x1251c81d0>, _source_notes=['Overall = pooled. Between = across unit means. Within = over time inside a unit (deviations from the unit mean, recentered on the grand mean).'], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x1251c8810>, _formats=[<great_tables._gt_data.FormatInfo object at 0x1251836d0>, <great_tables._gt_data.FormatInfo object at 0x125153b10>], _substitutions=[<great_tables._gt_data.FormatInfo object at 0x125151210>], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=Option

In [11]:
ex.explore_trend_plot_by_group(df, group_var="department", var="index_sdg").fig

### Panel models

Is the IDS associated with population size? Pooled OLS and the between estimator compare municipalities with each other; fixed effects compare each municipality with itself over time (with two periods, its change against its population growth); random effects mix both. The Hausman test asks whether random effects are consistent.

In [12]:
ex.analyze_panel_table(df, "index_sdg", ["ln_pop"]).etable

,Pooled OLS,Between,Fixed effects,Random effects
Dep. Variable,index_sdg,index_sdg,index_sdg,index_sdg
Estimator,PooledOLS,BetweenOLS,PanelOLS,RandomEffects
No. Observations,686,343,686,686
Cov. Est.,Clustered,Unadjusted,Clustered,Clustered
R-squared,0.1283,0.2002,0.2376,0.1244
R-Squared (Within),0.0545,0.0495,0.2376,0.0550
R-Squared (Between),0.1979,0.2002,-4.3761,0.1973
R-Squared (Overall),0.1283,0.1271,-4.3359,0.1283
F-statistic,100.70,85.374,106.60,97.150
P-value (F-stat),0.0000,0.0000,0.0000,0.0000


In [13]:
print(ex.analyze_hausman_test(df, "index_sdg", ["ln_pop"]).interpret())

Hausman test (χ²(1) = 83.308, p = 7.021e-20): **reject** the null — the random-effects assumption is violated, so prefer **fixed effects**. Note that failing to reject reflects a lack of evidence against random effects, not proof that it is correct.


Across municipalities, larger ones score higher: about 2.4 IDS points for each log point of population (pooled and between). The fixed-effects slope is far larger, but read it with care: with two periods and no year effects, it also absorbs the rise common to all municipalities, since both the IDS and population grew almost everywhere. The Hausman test rejects random effects. These are associations, not causal effects.

## 3. geometrics: clusters, convergence and inequality

`geo.for_geometrics()` returns the boundaries, the panel and its dictionary. Queen weights link municipalities that share a border; the one without neighbours is attached to its nearest.

In [14]:
gdf, sp, sp_dict = geo.for_geometrics("sdg", variables="index_sdg", groups=["department"])
sp = gm.set_labels(sp, sp_dict, set_panel=True)
w = gm.make_weights(gdf, method="queen")

### Spatial clustering in each period

In [15]:
for year in (2012, 2024):
    res = gm.explore_moran_plot(sp, "index_sdg", gdf=gdf, w=w, period=year)
    print(year, res.interpret().split("\n")[0], "\n")

2012 Global Moran's I for **index_sdg** in 2012 is 0.432, against an expectation of -0.00292 under spatial randomness — statistically significant at the 1% level (pseudo p = 0.001 from 999 permutations, under queen contiguity (1 island(s) attached to nearest neighbor), row-standardized, n=343). 

2024 Global Moran's I for **index_sdg** in 2024 is 0.468, against an expectation of -0.00292 under spatial randomness — statistically significant at the 1% level (pseudo p = 0.001 from 999 permutations, under queen contiguity (1 island(s) attached to nearest neighbor), row-standardized, n=343). 



In [16]:
gm.explore_lisa_cluster_map(sp, "index_sdg", gdf=gdf, w=w, period=2024).fig

High values cluster with high values (around La Paz, in the Cochabamba and Chuquisaca valleys and in the south, around Tarija and the Chaco), and low with low, in both periods; the clustering grew slightly stronger.

### β-convergence, with and without space

Growth between the periods regressed on the initial level: a negative slope means that the units that started lower grew faster. The spatial lag model separates a municipality's own catch-up from its neighbours'.

In [17]:
ols = gm.analyze_beta_convergence(sp, "index_sdg")
print(ols.interpret())

Across 343 units, the growth of **index_sdg** over a 12-period window is associated with its initial log level with a total slope of **-0.0287** (SE 0.00171), statistically significant at the 1% level.
The slope is negative — the β-convergence pattern: units that started lower tended to grow faster, narrowing initial gaps.
That slope implies a convergence speed of λ = 0.0351 per period and a half-life of about 19.7 periods — the time for half of an initial gap to close at this pace.

_These are associations, not causal effects. A causal reading needs a research design — see `explain('correlation_vs_causation')`._


In [18]:
sar = gm.analyze_beta_convergence(sp, "index_sdg", model="sar", gdf=gdf, w=w)
print(sar.interpret())

Across 343 units, the growth of **index_sdg** over a 12-period window is associated with its initial log level with a total slope of **-0.0345** (SE 0.00279), statistically significant at the 1% level.
The slope is negative — the β-convergence pattern: units that started lower tended to grow faster, narrowing initial gaps.
That slope implies a convergence speed of λ = 0.0445 per period and a half-life of about 15.6 periods — the time for half of an initial gap to close at this pace.
The SAR decomposition splits the total into a direct component of -0.0265 (own initial level) and a spillover (indirect) component of -0.00796 operating through neighboring units, under the weights: queen contiguity (1 island(s) attached to nearest neighbor), row-standardized, n=343.
The spatial-lag parameter ρ = 0.231 says each unit's growth moves together with its neighbors' growth, so part of the convergence pattern is shared across space rather than purely unit-by-unit.

_These are associations, not cau

### Inequality between and within departments

With two periods, σ-convergence (a trend in dispersion) cannot be estimated, but the Gini and Theil indices of each period can be compared, and the Theil index splits into a part between departments and a part within them.

In [19]:
print(gm.analyze_inequality_over_time(sp, "index_sdg").interpret())

Cross-sectional inequality in **index_sdg** is tracked across 343 units over 2 periods (2012 to 2024).

The Gini index fell from 0.065 to 0.0493 between the first and last period.

The Theil index fell from 0.00663 to 0.00381 between the first and last period.

The trend of the Gini index could not be estimated.

The trend of the Theil index could not be estimated.

_These are associations, not causal effects. A causal reading needs a research design — see `explain('correlation_vs_causation')`._


In [20]:
theil = gm.analyze_theil_decomposition(sp, "index_sdg", "department")
print(theil.interpret())
theil.fig

The Theil index of **index_sdg** splits additively into inequality **between** the 9 department groups and inequality **within** them (between + within = total, exactly).

In the latest period (2024), the total Theil index is 0.00381: about 15% of it lies between department groups and 85% within them — most inequality plays out among units inside the same department.

Over the window, the between-group share fell from 20% to 15% — group means are pulling closer together relative to the differences inside groups.

_These are associations, not causal effects. A causal reading needs a research design — see `explain('correlation_vs_causation')`._


## What we learned

- **geo4bolivia** found the variable, documented it, and prepared the panel, its change, its department aggregates and its boundaries in a few lines.
- **expdpy** split the IDS's variation almost evenly between differences across municipalities and change within them, and showed that cross-sectional and within-unit estimates of its association with population differ.
- **geometrics** showed that development clusters in space, that the municipalities that started lower gained more (β-convergence, half-life of about 20 years, with spillovers through neighbours in the spatial model), and that inequality fell; most of what remains lies within departments rather than between them.

Next: the [dynamics of the 15 goals](https://colab.research.google.com/github/quarcs-lab/geo4bolivia/blob/main/notebooks/sdg_dynamics.ipynb), the [geo4bolivia tutorial](https://quarcs-lab.github.io/geo4bolivia/tools/geo4bolivia-package/) and its [function reference](https://quarcs-lab.github.io/geo4bolivia/tools/geo4bolivia-reference/).